In [1]:
import json
from pathlib import Path
import random
import os

# Dataset path
TRAIN_DATASET_DIR = Path(f"{Path(os.getcwd()).parent}/geom_drugs_conformers/train")
TRAIN_MANIFEST_PATH = TRAIN_DATASET_DIR / "manifest.json"

TEST_DATASET_DIR = Path(f"{Path(os.getcwd()).parent}/geom_drugs_conformers/test")
TEST_MANIFEST_PATH = TEST_DATASET_DIR / "manifest.json"

conformer_train_dir = Path(f"{Path(os.getcwd()).parent}/sampling/geom_conformer_train")
conformer_test_dir = Path(f"{Path(os.getcwd()).parent}/sampling/geom_conformer_test")

conformer_train_dir.mkdir(parents=True, exist_ok=True)
conformer_test_dir.mkdir(parents=True, exist_ok=True)

In [2]:
# Load the manifest
with open(TRAIN_MANIFEST_PATH, 'r') as f:
    train_manifest = json.load(f)

with open(TEST_MANIFEST_PATH, 'r') as f:
    test_manifest = json.load(f)

print(f"Total entries in train manifest: {len(train_manifest)}")
print(f"Total entries in test manifest: {len(test_manifest)}")

Total entries in train manifest: 290945
Total entries in test manifest: 1000


In [3]:
def extract_smiles_from_manifest(manifest):     
    smiles_list = []
    failed = 0
    for entry in manifest:
        try:
            method = entry.get('structures', [])[0].get('method', '')
            if method.startswith('QM9:'):
                smiles = method[4:]  # Remove 'QM9:' prefix
                smiles_list.append(smiles)
            else:
                failed += 1
        except Exception as e:
            failed += 1
    print(f"Extracted {len(smiles_list)} SMILES strings")
    print(f"Failed to extract: {failed} entries")
    print(f"\nFirst 5 SMILES:")
    for i, smiles in enumerate(smiles_list[:5], 1):
        print(f"  {i}. {smiles[:80]}..." if len(smiles) > 80 else f"  {i}. {smiles}")
    return smiles_list, failed


print("Extracting SMILES from train manifest...")
train_smiles_list, train_failed = extract_smiles_from_manifest(train_manifest)
print("Extracting SMILES from test manifest...")
test_smiles_list, test_failed = extract_smiles_from_manifest(test_manifest)

train_smiles_set = set(train_smiles_list)
test_smiles_set = set(test_smiles_list)


Extracting SMILES from train manifest...
Extracted 290945 SMILES strings
Failed to extract: 0 entries

First 5 SMILES:
  1. CCOc1cc(C(=O)OCc2ccc(C(=O)OC)o2)cc(Br)c1OC
  2. Cc1nn2c(-c3cccnc3)ccnc2c1-c1ccccc1
  3. OC1(c2ccc3ccccc3c2)CC[NH+](Cc2ccccc2)C1
  4. CC1CN(CC(=O)Nc2ccc(Br)cn2)CC(C)O1
  5. CCCc1cnc(NCCNCC(O)COc2ccc(Cl)c(Cl)c2)nc1
Extracting SMILES from test manifest...
Extracted 1000 SMILES strings
Failed to extract: 0 entries

First 5 SMILES:
  1. CC(C)CCC(C(=O)O)C(C)CC(=O)NC1CCCCC1
  2. CCCN(CCC)CCCNC(=O)c1ccc2c(Cl)c3c(nc2c1)CCC3
  3. NCCS(=O)(=O)[NH2+]C1CCCCC1
  4. CCCCNC(=O)COC(=O)c1c2c(nc3ccccc13)CCC2
  5. CC(C)C[NH2+]c1c2c(nc3ccccc13)CCC2


In [4]:
# Some basic statistics
from collections import Counter

def print_smiles_statistics(smiles_list):
    print("SMILES Statistics:")
    print("=" * 60)
    print(f"Total unique SMILES: {len(set(smiles_list))}")
    print(f"Total SMILES (including duplicates): {len(smiles_list)}")
    print(f"Number of duplicates: {len(smiles_list) - len(set(smiles_list))}")

    # Check for duplicate SMILES
    smiles_counts = Counter(smiles_list)
    duplicates = {smiles: count for smiles, count in smiles_counts.items() if count > 1}
    if duplicates:
        print(f"\nFound {len(duplicates)} SMILES that appear multiple times:")
        for smiles, count in list(duplicates.items())[:5]:
            print(f"  '{smiles[:60]}...' appears {count} times")
    else:
        print("\nNo duplicate SMILES found (each SMILES appears exactly once)")

print_smiles_statistics(train_smiles_list)
print_smiles_statistics(test_smiles_list)

SMILES Statistics:
Total unique SMILES: 290939
Total SMILES (including duplicates): 290945
Number of duplicates: 6

Found 5 SMILES that appear multiple times:
  'C#CCOCCOCCOCCNc1nc(N2CCN(C(=O)[C@H](CCC(=O)O)n3cc(C(N)CO)nn3...' appears 2 times
  'COCCCOc1ccnc(CS(=O)c2nc3ccccc3[nH]2)c1C...' appears 2 times
  'C#CCOCCOCCOCCNc1nc(N2CCN(C(=O)[C@H](Cc3ccc(O)cc3)n3cc([C@@H]...' appears 2 times
  'NC(CCC(=O)NC(CS)C(=O)NCC(=O)O)C(=O)O...' appears 2 times
  'NCCC[C@H](N)CC(=O)N[C@H]1CNC(=O)[C@H](C2C[C@H](O)N=C(N)N2)NC...' appears 3 times
SMILES Statistics:
Total unique SMILES: 1000
Total SMILES (including duplicates): 1000
Number of duplicates: 0

No duplicate SMILES found (each SMILES appears exactly once)


In [6]:
random.seed(42)
train_smiles_sample = random.sample(sorted(train_smiles_set), 30)
test_smiles_sample = random.sample(sorted(test_smiles_set), 30)

In [7]:
# Write geom_drugs_sample SMILES to YAML file
import yaml
import hashlib

def write_smiles_yaml(smiles_sample, output_yaml_path):
    # Convert to sorted list of strings
    smiles_to_use = sorted([str(s) for s in smiles_sample])

    # Create tasks list in the required format
    tasks = []
    for i, smiles in enumerate(smiles_to_use):
        tasks.append({
            "task": "unconditional_smiles",
            "smiles": smiles,
            "num_samples": 5,
            "name": hashlib.sha256(smiles.encode()).hexdigest()
        })

    # Create the YAML structure``
    yaml_data = {
        "tasks": tasks
    }

    # Write to YAML file
    output_yaml_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_yaml_path, 'w') as f:
        yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

    print(f"Created YAML file with {len(tasks)} tasks from geom_drugs_sample")
    print(f"Saved to: {output_yaml_path.absolute()}")

train_output_yaml_path = conformer_train_dir / "smiles.yaml"
test_output_yaml_path = conformer_test_dir / "smiles.yaml"

write_smiles_yaml(train_smiles_sample, train_output_yaml_path)
write_smiles_yaml(test_smiles_sample, test_output_yaml_path)

Created YAML file with 30 tasks from geom_drugs_sample
Saved to: /datastor1/dy4652/proteinzen/sampling/geom_conformer_train/smiles.yaml
Created YAML file with 30 tasks from geom_drugs_sample
Saved to: /datastor1/dy4652/proteinzen/sampling/geom_conformer_test/smiles.yaml


In [ ]:
# Write geom_drugs_sample molecules to PDB files
import numpy as np
from rdkit import Chem
from rdkit.Chem.rdchem import RWMol, Conformer
from rdkit.Geometry import Point3D

def get_molecule_id_from_smiles(smiles: str, manifest: list):
    """Find molecule ID from SMILES in manifest."""
    for entry in manifest:
        method = entry.get('structure', {}).get('method', '')
        if method.startswith('QM9:'):
            entry_smiles = method[4:]  # Remove 'QM9:' prefix
            if entry_smiles == smiles:
                return entry['id']
    return None

def get_npz_paths(molecule_ids: str, dataset_dir: Path) -> [Path]:
    """Get path to .npz file from molecule ID."""
    # Subdirectory is characters at index 1-2 of the ID
    npz_paths = []
    for molecule_id in molecule_ids:
        subdir = molecule_id[1:3]
        npz_paths.append(dataset_dir / "structures" /  subdir / f"{molecule_id}.npz")
    return npz_paths

def write_geom_drugs_molecule_to_pdb(npz_paths: Path, output_dir):
    """Write geom_drugs_boltz molecule to PDB using RDKit."""
    # Load the .npz file
    output_paths = []

    for npz_path in npz_paths:
        data = np.load(npz_path)
        atoms = data['atoms']
        
        # Extract data
        Z = atoms['element'].astype(int)  # Atomic numbers
        q = atoms['charge'].astype(int)   # Formal charges
        xyz = atoms['coords'].astype(float)  # Coordinates
        
        n_atoms = len(Z)
        
        # Create RDKit molecule
        mol = RWMol()
        conf = Conformer(n_atoms)
        
        for i in range(n_atoms):
            atom = Chem.Atom(int(Z[i]))
            atom.SetFormalCharge(int(q[i]))
            idx = mol.AddAtom(atom)
            conf.SetAtomPosition(idx, Point3D(*xyz[i]))
        
        mol = mol.GetMol()
        conf.SetId(0)
        mol.AddConformer(conf, assignId=True)
        
        # Clear chirality and stereo (similar to QMugs function)
        from rdkit.Chem import rdchem
        for atom in mol.GetAtoms():
            atom.SetChiralTag(rdchem.ChiralType.CHI_UNSPECIFIED)
        for bond in mol.GetBonds():
            bond.SetStereo(rdchem.BondStereo.STEREONONE)
        
        output_path = output_dir / f"{npz_path.stem}.pdb"
        output_paths.append(output_path)
        Chem.MolToPDBFile(mol, str(output_path), confId=0)

    return output_paths

def write_conformer_pdb_paths(smiles_sample, manifest, output_dir, dataset_dir):

    smiles_to_use = sorted([str(s) for s in smiles_sample])
    # Build SMILES -> molecule ID mapping from manifest
    print("Building SMILES to molecule ID mapping...")
    smiles_to_ids = {}
    for entry in manifest:
        method = entry.get('structures', {})[0].get('method', '')
        if method.startswith('QM9:'):
            entry_smiles = method[4:]
            smiles_to_ids[entry_smiles] = entry['ids']
    
    print(f"Mapped {len(smiles_to_ids)} SMILES to molecule IDs\n")

    # Process each SMILES in geom_drugs_sample
    output_dir.mkdir(parents=True, exist_ok=True)

    first_conformer_output_pdb_paths = []
    for i, smiles in enumerate(smiles_to_use):
        smiles = str(smiles)
        
        if smiles not in smiles_to_ids:
            print(f"SMILES not found in manifest: {smiles[:60]}...")
            continue
        
        molecule_ids = smiles_to_ids[smiles]
        npz_paths = get_npz_paths(molecule_ids, dataset_dir)

        # if not npz_path.exists():
        #     print(f"Structure file not found: {npz_paths}")
        #     results[smiles] = None
        #     continue 
        try:
            first_conformer_output_pdb_paths.append(write_geom_drugs_molecule_to_pdb(
                npz_paths=npz_paths,
                output_dir=output_dir
            )[0])
            # print(f"✓ Wrote {output_path.name} ({len(np.load(npz_path)['atoms'])} atoms)")
        except Exception as e:
            print(f"Failed for {smiles[:60]}...")
            import traceback
            traceback.print_exc()

    return first_conformer_output_pdb_paths

write_conformer_pdb_paths(train_smiles_sample, train_manifest, conformer_train_dir / "conformer_mols", TRAIN_DATASET_DIR)
write_conformer_pdb_paths(test_smiles_sample, test_manifest, conformer_test_dir / "conformer_mols", TEST_DATASET_DIR)


Building SMILES to molecule ID mapping...
Mapped 290939 SMILES to molecule IDs

Building SMILES to molecule ID mapping...
Mapped 1000 SMILES to molecule IDs



In [18]:
import shutil
train_first_conformer_out_dir = conformer_train_dir / "first_conformer_mols"
test_first_conformer_out_dir = conformer_test_dir / "first_conformer_mols"

train_first_conformer_out_dir.mkdir(parents=True, exist_ok=True)
test_first_conformer_out_dir.mkdir(parents=True, exist_ok=True)

for pdb_path in train_first_conformer_output_pdb_paths:
    shutil.copy(pdb_path, train_first_conformer_out_dir / pdb_path.name)

for pdb_path in test_first_conformer_output_pdb_paths:
    shutil.copy(pdb_path, test_first_conformer_out_dir / pdb_path.name)


In [ ]:
# Write geom_drugs_sample SMILES to YAML file
import yaml

import glob
# Convert to sorted list of strings
def write_mol_yaml(first_conformer_output_pdb_paths, output_yaml):
    smiles_to_use = sorted([str(s) for s in geom_drugs_sample])

    # Create tasks list in the required format
    tasks = []
    for i in range(len(first_conformer_output_pdb_paths)):
        name = Path(first_conformer_output_pdb_paths[i]).stem
        name = name[:name.index("_")]
        tasks.append({
            "task": "unconditional_mol",
            "mol_pdb_path": str(first_conformer_output_pdb_paths[i]),
            "num_samples": 5,
            "name": name
        })

    # Create the YAML structure``
    yaml_data = {
        "tasks": tasks
    }

    # Write to YAML file
    output_yaml.parent.mkdir(parents=True, exist_ok=True)
    with open(output_yaml, 'w') as f:
        yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

    print(f"Created YAML file with {len(tasks)} tasks from geom_drugs_sample")
    print(f"Saved to: {output_yaml.absolute()}")

train_first_conformer_output_pdb_paths = glob.glob(str(conformer_train_dir / "first_conformer_mols" / "*.pdb"))
test_first_conformer_output_pdb_paths = glob.glob(str(conformer_test_dir / "first_conformer_mols" / "*.pdb"))

write_mol_yaml(train_first_conformer_output_pdb_paths, conformer_train_dir / "mol.yaml")
write_mol_yaml(test_first_conformer_output_pdb_paths, conformer_test_dir / "mol.yaml")

Created YAML file with 30 tasks from geom_drugs_sample
Saved to: /datastor1/dy4652/proteinzen/sampling/geom_conformer_train/mol.yaml
Created YAML file with 30 tasks from geom_drugs_sample
Saved to: /datastor1/dy4652/proteinzen/sampling/geom_conformer_test/mol.yaml
